<a href="https://colab.research.google.com/github/Not-kh-lily-23/dbank-longitudinal-prediction/blob/main/feature_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install pylangacq
import pandas as pd
import pylangacq
import os
from google.colab import drive
drive.mount('/content/drive')
base='/content/drive/MyDrive/DementiaBank Project'
mcsv=os.path.join(base,'cross_sectional_baseline_master.csv')
print("Loading baseline cohort data...")
df=pd.read_csv(mcsv)
df=df[df['cohort_status'].isin(['AD','Control'])].copy()
print(f"total transcripts: {len(df)}")
def extract_linguistic_features(filepath):
    try:
        chat=pylangacq.read_chat(filepath)
        par_utterances=[utt for utt in chat.utterances() if utt.participant=='PAR']
        if not par_utterances:
            return None
        tokens=[token for utt in par_utterances for token in utt.tokens]
        words=[t.word for t in tokens if t.word]
        word_count = len(words)
        if word_count==0:
            return None
        mlu_w=word_count/len(par_utterances)
        unique_words=set([w.lower() for w in words])
        ttr=len(unique_words)/word_count
        nouns=0
        pronouns=0
        for token in tokens:
            pos=str(token.pos).lower()
            if pos.startswith('n') and not pos.startswith('num'):
                nouns+=1
            elif pos.startswith('pro'):
                pronouns+=1
        pn_ratio=pronouns/(nouns+pronouns) if (nouns+pronouns)>0 else 0
        with open(filepath,'r',encoding='utf-8',errors='ignore') as f:
            raw_lines=f.readlines()
        par_raw_text=" ".join([line for line in raw_lines if line.startswith('*PAR:')])
        import re
        filled_pauses=len(re.findall(r'&[-\w]+',par_raw_text))
        dfl_rate=(filled_pauses/word_count)*100 if word_count>0 else 0
        return {
            'word_count': word_count,
            'mlu_w': mlu_w,
            'ttr': ttr,
            'pn_ratio': pn_ratio,
            'disfluency_rate': dfl_rate
        }
    except Exception as e:
        print(f"failed to process {filepath}: {e}")
        return None
print("extracting")
f_list=[]
for index,row in df.iterrows():
    fpath=row['file_path']
    if not os.path.exists(fpath):
        continue
    feats=extract_linguistic_features(fpath)
    if feats:
        combined_data={**row.to_dict(),**feats}
        f_list.append(combined_data)
feat_df=pd.DataFrame(f_list)
op=os.path.join(base,'nlp_features_final.csv')
feat_df.to_csv(op,index=False)
print(f"extracted features for {len(feat_df)} baseline transcripts.")
print(f"file saved to: {op}")
display(feat_df[['participant_id','cohort_status','mlu_w','ttr','pn_ratio','disfluency_rate']].head(5))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading baseline cohort data...
total transcripts: 266
extracting
extracted features for 266 baseline transcripts.
file saved to: /content/drive/MyDrive/DementiaBank Project/nlp_features_final.csv


,participant_id,cohort_status,mlu_w,ttr,pn_ratio,disfluency_rate
0,1,AD,8.600000,0.593023,0.304348,4.651163
1,2,Control,8.388889,0.556291,0.244444,3.311258
2,3,AD,9.304348,0.429907,0.473684,4.205607
3,5,AD,6.666667,0.633333,0.266667,6.666667
4,6,Control,7.357143,0.572816,0.333333,4.854369
